# Model-Based CF — Matrix Factorization (SVD via SGD)

Train a latent-factor model from scratch with NumPy and evaluate:
- Hyperparameter sweep: n_factors ∈ {10, 50, 100}, lr ∈ {0.001, 0.005, 0.01}
- Convergence curves (train vs. val RMSE per epoch)
- Same evaluation metrics as notebook 02 for a fair comparison

In [ ]:
import sys, os

# Works both locally (run from notebooks/) and on Google Colab (run from repo root)
_src = os.path.join('..', 'src') if os.path.isdir(os.path.join('..', 'src')) else 'src'
_data = os.path.join('..', 'data', 'processed') if os.path.isdir(os.path.join('..', 'data')) else os.path.join('data', 'processed')
sys.path.insert(0, _src)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from data_loader import get_data
from model_based.matrix_factorization import MatrixFactorization
from evaluation.metrics import evaluate_predictions, evaluate_ranking, catalog_coverage

sns.set_theme(style='whitegrid')
%matplotlib inline

In [ ]:
train, test, matrix, movies = get_data()
ALL_ITEMS = list(matrix.columns)
N_ITEMS = len(ALL_ITEMS)
K = 10
RELEVANCE_THRESHOLD = 4

test_relevant = (
    test[test['rating'] >= RELEVANCE_THRESHOLD]
    .groupby('user_id')['item_id']
    .apply(set)
    .to_dict()
)

train_rated = train.groupby('user_id')['item_id'].apply(set).to_dict()

## 1. Single training run with convergence plot

In [ ]:
mf = MatrixFactorization(n_factors=50, n_epochs=30, lr=0.005, reg=0.02)
mf.fit(train, val=test)

epochs = range(1, len(mf.train_loss_) + 1)
plt.figure(figsize=(8, 4))
plt.plot(epochs, mf.train_loss_, label='Train RMSE')
plt.plot(epochs, mf.val_loss_, label='Val RMSE')
plt.xlabel('Epoch')
plt.ylabel('RMSE')
plt.title('Training Convergence (n_factors=50, lr=0.005)')
plt.legend()
plt.tight_layout()
plt.show()

## 2. Hyperparameter sweep

In [ ]:
sweep_results = []

for n_factors in [10, 50, 100]:
    for lr in [0.001, 0.005, 0.01]:
        print(f'n_factors={n_factors}  lr={lr} ...', end=' ')
        model = MatrixFactorization(n_factors=n_factors, n_epochs=20, lr=lr, reg=0.02)
        model.fit(train)

        test['predicted'] = model.predict_batch(test)
        pred_metrics = evaluate_predictions(test, pred_col='predicted')

        recs = {}
        for uid in test['user_id'].unique():
            rated = train_rated.get(uid, set())
            recs[uid] = model.recommend(uid, ALL_ITEMS, rated_item_ids=rated, n=K)

        rank_metrics = evaluate_ranking(recs, test_relevant, k=K)
        cov = catalog_coverage(list(recs.values()), N_ITEMS)

        sweep_results.append({'n_factors': n_factors, 'lr': lr,
                               **pred_metrics, **rank_metrics, 'Coverage': cov})
        print(f"RMSE={pred_metrics['RMSE']:.4f}  NDCG@10={rank_metrics['NDCG@10']:.4f}")

sweep_df = pd.DataFrame(sweep_results)
sweep_df.sort_values('RMSE')

## 3. RMSE heatmap: n_factors vs lr

In [ ]:
pivot = sweep_df.pivot(index='n_factors', columns='lr', values='RMSE')
plt.figure(figsize=(7, 4))
sns.heatmap(pivot, annot=True, fmt='.4f', cmap='YlOrRd_r')
plt.title('RMSE — n_factors vs Learning Rate')
plt.tight_layout()
plt.show()

## 4. Best model — full evaluation

In [ ]:
best_row = sweep_df.loc[sweep_df['RMSE'].idxmin()]
print(f'Best config: n_factors={int(best_row.n_factors)}  lr={best_row.lr}')

best_mf = MatrixFactorization(
    n_factors=int(best_row.n_factors),
    n_epochs=30,
    lr=float(best_row.lr),
    reg=0.02
)
best_mf.fit(train, val=test)

test['predicted'] = best_mf.predict_batch(test)
pred_metrics = evaluate_predictions(test, pred_col='predicted')

recs = {}
for uid in test['user_id'].unique():
    rated = train_rated.get(uid, set())
    recs[uid] = best_mf.recommend(uid, ALL_ITEMS, rated_item_ids=rated, n=K)

rank_metrics = evaluate_ranking(recs, test_relevant, k=K)
cov = catalog_coverage(list(recs.values()), N_ITEMS)

mf_best = {'Model': 'Matrix Factorization', **pred_metrics, **rank_metrics, 'Coverage': cov}
pd.DataFrame([mf_best])

In [ ]:
# Save for comparison notebook
pd.DataFrame([mf_best]).to_csv(os.path.join(_data, 'mf_best_results.csv'), index=False)

# Save sweep for heatmap reuse
sweep_df.to_csv(os.path.join(_data, 'mf_sweep_results.csv'), index=False)

## 5. Sample recommendations

In [ ]:
SAMPLE_USER = 1
rated = train_rated.get(SAMPLE_USER, set())
top10 = best_mf.recommend(SAMPLE_USER, ALL_ITEMS, rated_item_ids=rated, n=10)

print(f'Top-10 recommendations for user {SAMPLE_USER} (Matrix Factorization):')
movies[movies['item_id'].isin(top10)][['item_id', 'title']]